In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", 'vllm==0.32.0 peft')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'):
        sh(os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'common.py': '"""Shared helpers for GreenScan training on Kaggle (T4) and on a rented A100 (vast.ai).\n\nData is looked up in $GS_DATA, ./data, ../data, then /kaggle/input (a private dataset). Outputs go to $GS_OUT\n(default ./out). Every script prints "[cell] ..." at each stage and flushes, so a log shows where a run is.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nimport re\nimport time\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef cell(name: str) -> None:\n    print(f"[cell] {name}", flush=True)\n\n\ndef log(*a) -> None:\n    print(*a, flush=True)\n\n\ndef find(name: str) -> Path:\n    for root in (os.environ.get("GS_DATA", ""), "data", "../data", "/kaggle/input"):\n        if root and Path(root).exists():\n            hit = next(iter(sorted(Path(root).rglob(name))), None)\n            if hit:\n                return hit\n    raise FileNotFoundError(name)\n\n\ndef has(name: str) -> bool:\n    try:\n        find(name)\n        return True\n    except FileNotFoundError:\n        return False\n\n\ndef jl(path) -> list[dict]:\n    p = Path(path) if not isinstance(path, str) or Path(path).exists() else find(path)\n    return [json.loads(x) for x in open(p, encoding="utf-8") if x.strip()]\n\n\ndef wjl(path, rows) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    with open(path, "w", encoding="utf-8") as fh:\n        for r in rows:\n            fh.write(json.dumps(r, ensure_ascii=False) + "\\n")\n\n\ndef out_dir(sub: str = "") -> Path:\n    p = Path(os.environ.get("GS_OUT", "out")) / sub\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef save_json(path, obj) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")\n\n\ndef seed_all(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n    except ImportError:\n        pass\n\n\ndef norm60(s: str) -> str:\n    return re.sub(r"\\s+", " ", (s or "").lower()).strip()[:60]\n\n\ndef company_folds(groups: list[str], k: int = 5, seed: int = 13) -> list[tuple[list[int], list[int]]]:\n    """Out-of-company folds: every company sits in exactly one test fold; folds balanced by row count."""\n    count = Counter(groups)\n    comps = sorted(count, key=lambda c: (-count[c], c))\n    rng = random.Random(seed)\n    rng.shuffle(comps)\n    comps.sort(key=lambda c: -count[c])\n    load, fold_of = [0] * k, {}\n    for c in comps:\n        f = min(range(k), key=lambda i: load[i])\n        fold_of[c] = f\n        load[f] += count[c]\n    out = []\n    for f in range(k):\n        test = [i for i, g in enumerate(groups) if fold_of[g] == f]\n        train = [i for i, g in enumerate(groups) if fold_of[g] != f]\n        if test and train:\n            out.append((train, test))\n    return out\n\n\ndef prf(gold: list[str], pred: list[str], labels: list[str]) -> dict:\n    per = {}\n    for l in labels:\n        tp = sum(g == l and p == l for g, p in zip(gold, pred))\n        fp = sum(g != l and p == l for g, p in zip(gold, pred))\n        fn = sum(g == l and p != l for g, p in zip(gold, pred))\n        pr = tp / (tp + fp) if tp + fp else 0.0\n        rc = tp / (tp + fn) if tp + fn else 0.0\n        per[l] = {"precision": round(pr, 4), "recall": round(rc, 4),\n                  "f1": round(2 * pr * rc / (pr + rc), 4) if pr + rc else 0.0, "support": tp + fn, "predicted": tp + fp}\n    present = [l for l in labels if per[l]["support"]]\n    macro = sum(per[l]["f1"] for l in present) / len(present) if present else 0.0\n    acc = sum(g == p for g, p in zip(gold, pred)) / len(gold) if gold else 0.0\n    return {"macro_f1": round(macro, 4), "accuracy": round(acc, 4), "per_class": per, "n": len(gold)}\n\n\ndef confusion(gold: list[str], pred: list[str], labels: list[str]) -> list[list[int]]:\n    idx = {l: i for i, l in enumerate(labels)}\n    m = [[0] * len(labels) for _ in labels]\n    for g, p in zip(gold, pred):\n        if g in idx and p in idx:\n            m[idx[g]][idx[p]] += 1\n    return m\n\n\ndef boot_ci(items: list, groups: list[str], stat, n: int = 1000, seed: int = 7) -> list[float] | None:\n    """95% interval of stat(items) resampling whole companies (cluster bootstrap)."""\n    by = defaultdict(list)\n    for it, g in zip(items, groups):\n        by[g].append(it)\n    keys = sorted(by)\n    if len(keys) < 2:\n        return None\n    rng = random.Random(seed)\n    vals = []\n    for _ in range(n):\n        sample = [it for _ in keys for it in by[rng.choice(keys)]]\n        try:\n            vals.append(stat(sample))\n        except ZeroDivisionError:\n            continue\n    vals.sort()\n    return [round(vals[int(0.025 * len(vals))], 4), round(vals[int(0.975 * len(vals)) - 1], 4)] if vals else None\n\n\nclass Timer:\n    def __init__(self):\n        self.t0 = time.time()\n\n    def __call__(self) -> float:\n        return round(time.time() - self.t0, 1)\n',
    'guards.py': '"""Checks applied in code to every model reply before it is used (registration llm_stages_v1).\n\nA model may read and may phrase; it may not invent. These are the two ways it could, and both are caught here\nrather than trusted to the prompt.\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\nimport unicodedata\n\n_NUM = re.compile(r"\\d+(?:[.,]\\d+)*")\n\n\ndef fold(text: str) -> str:\n    text = unicodedata.normalize("NFC", text or "").lower()\n    return re.sub(r"\\s+", " ", text).strip()\n\n\ndef numbers(text: str) -> set[str]:\n    """Digits as written, with separators removed: \'1.250.000\' and \'1,250,000\' are the same number."""\n    return {re.sub(r"[.,]", "", m) for m in _NUM.findall(text or "")}\n\n\ndef new_numbers(reply: str, *sources: str) -> set[str]:\n    """Numbers in `reply` that no source writes. Non-empty means the reply is rejected."""\n    allowed: set[str] = set()\n    for s in sources:\n        allowed |= numbers(s)\n    return numbers(reply) - allowed\n\n\ndef verbatim(span, sentence: str) -> str | None:\n    """`span` if the sentence writes it (case, spacing and Unicode form ignored), else None."""\n    if not isinstance(span, str) or not span.strip():\n        return None\n    return span.strip() if fold(span) in fold(sentence) else None\n\n\ndef parse_json(text: str | None) -> dict | None:\n    """First JSON object in a reply (fences and text around it ignored)."""\n    if not text:\n        return None\n    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)\n    start = text.find("{")\n    while start != -1:\n        depth = 0\n        for i in range(start, len(text)):\n            depth += {"{": 1, "}": -1}.get(text[i], 0)\n            if depth == 0:\n                try:\n                    obj = json.loads(text[start:i + 1])\n                    return obj if isinstance(obj, dict) else None\n                except json.JSONDecodeError:\n                    break\n        start = text.find("{", start + 1)\n    return None\n',
    'gen_teacher.py': '"""Teacher explanations for distillation: a large model writes the explanation of each stored verdict.\n\n    python -u gen_teacher.py --model Qwen/Qwen3-32B-AWQ --tp 2 --input explain_train.jsonl --field msgs_large \\\n        --out out/teacher/explain_teacher.jsonl [--limit N]\n\nGreedy decoding, thinking off. Keeps an answer only when it parses as {"explanation", "next_action"}, is 12-140 words,\nand adds no number absent from the input (guards.new_numbers, the product\'s own check). Kept rows carry the student\nprompt (msgs_small) and the teacher answer as the SFT target.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport sys\nfrom collections import Counter\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import Timer, cell, jl, log, save_json, wjl  # noqa: E402\nfrom guards import new_numbers  # noqa: E402\n\n\ndef parse(text: str) -> dict | None:\n    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()\n    m = re.search(r"\\{.*\\}", text, flags=re.S)\n    if not m:\n        return None\n    try:\n        d = json.loads(m.group(0))\n    except json.JSONDecodeError:\n        return None\n    if not isinstance(d, dict) or not str(d.get("explanation") or "").strip():\n        return None\n    return {"explanation": str(d["explanation"]).strip(), "next_action": str(d.get("next_action") or "").strip()}\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", required=True)\n    ap.add_argument("--tp", type=int, default=1)\n    ap.add_argument("--input", default="explain_train.jsonl")\n    ap.add_argument("--field", default="msgs_large")\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--limit", type=int, default=0)\n    ap.add_argument("--max-len", type=int, default=6144)\n    ap.add_argument("--max-tokens", type=int, default=450)\n    args = ap.parse_args()\n\n    cell("teacher load")\n    rows = jl(args.input)[: args.limit or None]\n    from transformers import AutoTokenizer\n    from vllm import LLM, SamplingParams\n    tok = AutoTokenizer.from_pretrained(args.model)\n    prompts = [tok.apply_chat_template(r[args.field], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n               for r in rows]\n    t = Timer()\n    llm = LLM(model=args.model, tensor_parallel_size=args.tp, max_model_len=args.max_len, gpu_memory_utilization=0.92,\n              enforce_eager=False, seed=0)\n    log(f"[teacher] loaded {args.model} in {t()} s; {len(rows)} prompts")\n\n    cell("teacher generate")\n    t = Timer()\n    outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=args.max_tokens))\n    secs = t()\n    why = Counter()\n    kept, raw = [], []\n    for r, o in zip(rows, outs):\n        text = o.outputs[0].text\n        d = parse(text)\n        user = r[args.field][-1]["content"]\n        reason = "kept"\n        if d is None:\n            reason = "unparsed"\n        else:\n            words = len((d["explanation"] + " " + d["next_action"]).split())\n            if not 12 <= words <= 140:\n                reason = "length"\n            elif new_numbers(d["explanation"] + " " + d["next_action"], user):\n                reason = "new_number"\n        why[reason] += 1\n        raw.append({"id": r["id"], "reason": reason, "text": text})\n        if reason == "kept":\n            kept.append({"id": r["id"], "company": r.get("company"), "verdict": r.get("verdict"),\n                         "messages": r["msgs_small"], "target": json.dumps(d, ensure_ascii=False)})\n    out = Path(args.out)\n    wjl(out, kept)\n    wjl(out.with_name(out.stem + "_raw.jsonl"), raw)\n    ntok = sum(len(o.outputs[0].token_ids) for o in outs)\n    save_json(out.with_name(out.stem + "_summary.json"),\n              {"model": args.model, "n": len(rows), "reasons": dict(why), "seconds": secs,\n               "output_tokens": ntok, "tokens_per_second": round(ntok / max(secs, 1e-6), 1)})\n    log(f"[teacher] {dict(why)} in {secs} s, {ntok} output tokens")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'train_llm_sft.py': '"""LoRA supervised fine-tuning of a chat model: loss = token cross-entropy on the answer only (prompt masked).\n\n    python -u train_llm_sft.py --base Qwen/Qwen3-4B-Instruct-2507 --train out/teacher/explain_teacher.jsonl \\\n        --name explain-4b --epochs 2 --dtype fp16 [--merge]          # Kaggle T4\n    python -u train_llm_sft.py --base Qwen/Qwen3-14B --train data/stance_sft.jsonl --name stance-14b --dtype bf16  # A100\n\nInput rows: {"id", "company", "messages": [system, user], "target": "<assistant answer>"}. About 10% of companies are\nheld out for the validation loss (out-of-company). Writes out/<name>/adapter (+ merged/ with --merge) and\nout/<name>/history.json (train loss every 10 steps, validation loss every epoch) for the charts.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport math\nimport random\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import Timer, cell, jl, log, out_dir, save_json, seed_all  # noqa: E402\n\n\ndef encode(tok, row, max_len):\n    prompt = tok.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n    p = tok(prompt, add_special_tokens=False)["input_ids"]\n    a = tok(row["target"] + (tok.eos_token or ""), add_special_tokens=False)["input_ids"]\n    if len(p) + len(a) > max_len:\n        return None\n    return {"input_ids": p + a, "labels": [-100] * len(p) + a}\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--train", required=True)\n    ap.add_argument("--name", required=True)\n    ap.add_argument("--epochs", type=float, default=2)\n    ap.add_argument("--lr", type=float, default=2e-4)\n    ap.add_argument("--r", type=int, default=16)\n    ap.add_argument("--alpha", type=int, default=32)\n    ap.add_argument("--max-len", type=int, default=2048)\n    ap.add_argument("--accum", type=int, default=8)\n    ap.add_argument("--dtype", choices=["fp16", "bf16"], default="fp16")\n    ap.add_argument("--val-frac", type=float, default=0.1)\n    ap.add_argument("--merge", action="store_true")\n    ap.add_argument("--merge-dir", default="", help="where the merged fp16 model goes (Kaggle: /tmp, outside the 20 GB output)")\n    ap.add_argument("--seed", type=int, default=13)\n    args = ap.parse_args()\n    seed_all(args.seed)\n\n    cell(f"sft load {args.base}")\n    import torch\n    from peft import LoraConfig, get_peft_model\n    from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup\n    dtype = torch.float16 if args.dtype == "fp16" else torch.bfloat16\n    tok = AutoTokenizer.from_pretrained(args.base)\n    rows = jl(args.train)\n    comps = sorted({r.get("company") or "?" for r in rows})\n    random.Random(args.seed).shuffle(comps)\n    val_comps = set(comps[: max(1, round(len(comps) * args.val_frac))]) if len(comps) > 3 else set()\n    enc = [(r, encode(tok, r, args.max_len)) for r in rows]\n    dropped = sum(e is None for _, e in enc)\n    train = [e for r, e in enc if e and (r.get("company") or "?") not in val_comps]\n    val = [e for r, e in enc if e and (r.get("company") or "?") in val_comps]\n    log(f"[sft] rows {len(rows)} · train {len(train)} · val {len(val)} (companies {sorted(val_comps)}) · "\n        f"too long {dropped}")\n\n    model = AutoModelForCausalLM.from_pretrained(args.base, torch_dtype=dtype, device_map={"": 0})\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    model.enable_input_require_grads()\n    model = get_peft_model(model, LoraConfig(r=args.r, lora_alpha=args.alpha, lora_dropout=0.05, task_type="CAUSAL_LM",\n                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj",\n                                                             "gate_proj", "up_proj", "down_proj"]))\n    model.print_trainable_parameters()\n    params = [p for p in model.parameters() if p.requires_grad]\n    steps = math.ceil(len(train) * args.epochs / args.accum)\n    opt = torch.optim.AdamW(params, lr=args.lr, weight_decay=0.0)\n    sch = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * steps)), steps)\n    scaler = torch.amp.GradScaler(enabled=args.dtype == "fp16")\n    hist = {"train": [], "val": [], "config": vars(args), "train_rows": len(train), "val_rows": len(val)}\n\n    def loss_of(e):\n        ids = torch.tensor([e["input_ids"]], device="cuda")\n        lab = torch.tensor([e["labels"]], device="cuda")\n        with torch.autocast("cuda", dtype=dtype):\n            return model(input_ids=ids, labels=lab).loss\n\n    def evaluate():\n        if not val:\n            return None\n        model.eval()\n        with torch.no_grad():\n            v = [loss_of(e).float().item() for e in val]\n        model.train()\n        return round(sum(v) / len(v), 4)\n\n    cell("sft train")\n    t, step, seen, run, bad = Timer(), 0, 0, [], 0\n    hist["val"].append({"epoch": 0, "loss": evaluate()})\n    model.train()\n    total = int(len(train) * args.epochs)\n    order = []\n    while len(order) < total:\n        ep = list(range(len(train)))\n        random.shuffle(ep)\n        order += ep\n    for i, j in enumerate(order[:total], 1):\n        loss = loss_of(train[j]) / args.accum\n        if not torch.isfinite(loss):\n            bad += 1\n            if bad >= 5:\n                raise SystemExit("[sft] loss is not finite 5 times: try --dtype bf16 (A100) or a smaller --lr")\n            opt.zero_grad(set_to_none=True)\n            continue\n        scaler.scale(loss).backward()\n        run.append(loss.item() * args.accum)\n        if i % args.accum == 0 or i == total:\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(params, 1.0)\n            scaler.step(opt)\n            scaler.update()\n            opt.zero_grad(set_to_none=True)\n            sch.step()\n            step += 1\n            if step % 10 == 0 or i == total:\n                hist["train"].append({"step": step, "epoch": round(i / len(train), 3), "loss": round(sum(run) / len(run), 4),\n                                      "lr": sch.get_last_lr()[0]})\n                log(f"[sft] step {step}/{steps} epoch {i / len(train):.2f} loss {sum(run) / len(run):.4f} · {t()} s")\n                run = []\n        if i % len(train) == 0 and i < total:\n            v = evaluate()\n            hist["val"].append({"epoch": round(i / len(train), 2), "loss": v})\n            log(f"[sft] epoch {i / len(train):.0f} val loss {v}")\n    hist["val"].append({"epoch": args.epochs, "loss": evaluate()})\n    hist["seconds"] = t()\n    log(f"[sft] val losses {hist[\'val\']}")\n\n    cell("sft save")\n    od = out_dir(args.name)\n    model.save_pretrained(od / "adapter")\n    tok.save_pretrained(od / "adapter")\n    save_json(od / "history.json", hist)\n    if args.merge:\n        mdir = Path(args.merge_dir) if args.merge_dir else od / "merged"\n        merged = model.merge_and_unload()\n        merged.save_pretrained(mdir, safe_serialization=True)\n        tok.save_pretrained(mdir)\n        log(f"[sft] merged model -> {mdir}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'eval_explain.py': '"""Score explanation models on the registered S6 set (142 items): must-mention proxy, new numbers, length, speed.\n\n    python -u eval_explain.py --base Qwen/Qwen3-4B-Instruct-2507 --adapter out/explain-4b/adapter --field msgs_small\n    python -u eval_explain.py --base Qwen/Qwen3-32B-AWQ --tp 2 --field msgs_large --tag teacher-32b\n\nSystems scored in one vLLM process: the base model, and the base + LoRA adapter when --adapter is given. The scoring\nis benchmark/llm_stages/score.py score_s6 (same covered() rule, same new_numbers guard, company-cluster bootstrap).\nWrites out/explain_eval/<tag>.json and the raw answers.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import Timer, boot_ci, cell, jl, log, out_dir, save_json, wjl  # noqa: E402\nfrom gen_teacher import parse  # noqa: E402\nfrom guards import fold, new_numbers  # noqa: E402\n\n\ndef covered(phrase, text):\n    words = [w for w in fold(phrase).split() if len(w) > 2]\n    return bool(words) and sum(w in fold(text) for w in words) / len(words) >= 0.6\n\n\ndef score(data, answers):\n    rows, bad, empty, unparsed = [], 0, 0, 0\n    for d in data:\n        p = answers.get(d["id"])\n        unparsed += p is None\n        text = " ".join(str((p or {}).get(k, "")) for k in ("explanation", "next_action")).strip()\n        empty += not text\n        nn = new_numbers(text, d["claim"], d["rationale"],\n                         *[f"{e.get(\'source_name\')} {e.get(\'page\')} {e.get(\'text\') or \'\'}" for e in d["evidence"]])\n        bad += bool(nn)\n        mm = d["must_mention"]\n        rows.append({"company": d["company"], "cov": (sum(covered(m, text) for m in mm) / len(mm)) if mm else None,\n                     "words": len(text.split()), "new_number": bool(nn)})\n\n    def cov(rs):\n        v = [r["cov"] for r in rs if r["cov"] is not None]\n        return sum(v) / len(v) if v else 0.0\n\n    return {"n": len(rows), "must_mention_proxy": round(cov(rows), 3),\n            "ci95": boot_ci(rows, [r["company"] for r in rows], cov), "with_new_number": bad, "empty": empty,\n            "unparsed": unparsed, "words_mean": round(sum(r["words"] for r in rows) / len(rows), 1)}\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--adapter", default="")\n    ap.add_argument("--tp", type=int, default=1)\n    ap.add_argument("--field", default="msgs_small")\n    ap.add_argument("--tag", default="")\n    ap.add_argument("--max-len", type=int, default=6144)\n    args = ap.parse_args()\n    cell("explain eval load")\n    data = jl("s6_explain.jsonl")\n    from transformers import AutoTokenizer\n    from vllm import LLM, SamplingParams\n    tok = AutoTokenizer.from_pretrained(args.base)\n    prompts = [tok.apply_chat_template(d[args.field], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n               for d in data]\n    kw = dict(enable_lora=True, max_lora_rank=64) if args.adapter else {}\n    llm = LLM(model=args.base, tensor_parallel_size=args.tp, max_model_len=args.max_len, gpu_memory_utilization=0.9,\n              seed=0, **kw)\n    sp = SamplingParams(temperature=0.0, max_tokens=400)\n    systems = [("base", None)]\n    if args.adapter:\n        from vllm.lora.request import LoRARequest\n        systems.append(("lora", LoRARequest("explain", 1, args.adapter)))\n    od = out_dir("explain_eval")\n    tag = args.tag or Path(args.base).name\n    result = {"base": args.base, "adapter": args.adapter, "field": args.field, "systems": {}}\n    for name, lora in systems:\n        cell(f"explain eval generate {name}")\n        t = Timer()\n        outs = llm.generate(prompts, sp, lora_request=lora)\n        secs = t()\n        answers = {d["id"]: parse(o.outputs[0].text) for d, o in zip(data, outs)}\n        wjl(od / f"{tag}_{name}_answers.jsonl",\n            [{"id": d["id"], "text": o.outputs[0].text, "parsed": answers[d["id"]]} for d, o in zip(data, outs)])\n        s = score(data, answers)\n        s["seconds"] = secs\n        s["seconds_per_item_batched"] = round(secs / len(data), 3)\n        result["systems"][name] = s\n        log(f"[explain] {tag} {name}: {s}")\n    save_json(od / f"{tag}.json", result)\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'export_gguf.sh': '#!/usr/bin/env bash\n# Merged HF model -> GGUF Q4_K_M + Ollama Modelfile, for the offline laptop (Ollama).\n#   bash export_gguf.sh /tmp/explain-4b-merged out/explain-4b-gguf greenscan-explain-4b\nset -euo pipefail\nMERGED="$1"; OUT="$2"; NAME="${3:-greenscan-model}"\necho "[cell] gguf build llama.cpp"\nmkdir -p "$OUT"\n[ -d /tmp/llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp\npip install -q /tmp/llama.cpp/gguf-py sentencepiece protobuf\ncmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF > /tmp/cmake.log\ncmake --build /tmp/llama.cpp/build --target llama-quantize -j "$(nproc)" > /tmp/build.log\necho "[cell] gguf convert f16"\npython -u /tmp/llama.cpp/convert_hf_to_gguf.py "$MERGED" --outfile /tmp/model-f16.gguf --outtype f16\necho "[cell] gguf quantize Q4_K_M"\n/tmp/llama.cpp/build/bin/llama-quantize /tmp/model-f16.gguf "$OUT/$NAME-q4_k_m.gguf" Q4_K_M\nrm -f /tmp/model-f16.gguf\ncat > "$OUT/Modelfile" <<EOF\nFROM ./$NAME-q4_k_m.gguf\nPARAMETER temperature 0\nPARAMETER num_ctx 4096\nEOF\nls -la "$OUT"\necho "[gguf] done: ollama create $NAME -f $OUT/Modelfile"\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: python -u train_llm_sft.py --base Qwen/Qwen3-1.7B --train ex', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u train_llm_sft.py --base Qwen/Qwen3-1.7B --train explain_teacher.jsonl --name explain-1p7b --epochs 2 --dtype fp16 --merge --merge-dir /tmp/explain-1p7b-merged')


In [ ]:
print('[cell] run 2: python -u eval_explain.py --base Qwen/Qwen3-1.7B --adapter o', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u eval_explain.py --base Qwen/Qwen3-1.7B --adapter out/explain-1p7b/adapter --field msgs_small --tag student-1p7b')


In [ ]:
print('[cell] run 3: bash export_gguf.sh /tmp/explain-1p7b-merged out/explain-1p7', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('bash export_gguf.sh /tmp/explain-1p7b-merged out/explain-1p7b-gguf greenscan-explain-1p7b')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)